## 1. Setup

In [ ]:
import pandas as pd
from pathlib import Path

SIMULATED_DIR = Path("../data/simulated")

## 2. Load the data

In [ ]:
ehr_raw = pd.read_csv(SIMULATED_DIR / "ehr_providers.csv")
hr_raw = pd.read_csv(SIMULATED_DIR / "hr_providers.csv")
cred_raw = pd.read_csv(SIMULATED_DIR / "credentialing_providers.csv")

## 3. Set up findings log and helper functions

In [ ]:
findings = pd.DataFrame(columns=[
    "dataset",
    "column",
    "issue",
    "example",
    "planned_action",    
    "status"
])

# Helper function to add log
def add_finding(dataset, column, issue, example, planned_action):
    duplicate = (
        (findings["dataset"] == dataset) &
        (findings["column"] == column) &
        (findings["issue"] == issue)
    ).any()
    
    if duplicate:
        print("Finding already exists.")
        return
    
    findings.loc[len(findings)] = {
        "dataset": dataset,
        "column": column,
        "issue": issue,
        "example": example,
        "planned_action": planned_action,
        "status": "Open"
    }

# Helper function to delete log
def delete_finding(dataset, column, issue):
    global findings
    
    findings = findings[
        ~(
            (findings["dataset"] == dataset) &
            (findings["column"] == column) &
            (findings["issue"] == issue)
        )
    ].reset_index(drop=True)

## 4. Inspecting the datasets

In [ ]:
simulated_providers = {
    "EHR" : ehr_raw,
    "HR" : hr_raw,
    "CREDENTIALING" : cred_raw
}

# Checking columns and df shape
for name, df in simulated_providers.items():
    print(name)
    print(df.shape)
    print(df.columns)
    print('\n')

## 5. Validating column datatypes

In [ ]:
for name, df in simulated_providers.items():
    print(name)
    print(df.dtypes)
    print('\n')

# NPI stored as float across EHR, HR, and CREDENTIALING
# Convert to string and validate 10 digits
for dataset in ["EHR", "HR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="npi",
        issue="NPI inferred as numeric during import",
        example="1234567890.0",
        planned_action="Load/convert as string and validate 10-digit format"
    )

# Zip stored as int across EHR and CREDENTIALING
# Convert to string 
for dataset in ["EHR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="zip",
        issue="Zip inferred as numeric during import",
        example="773386118",
        planned_action="Convert to string"
    )

In [ ]:
cred_raw.head()

# Phone stored as float in CREDENTIALING
# Convert to string and validate
add_finding(
        dataset="Credentialing",
        column="phone",
        issue="phone stored as float",
        example="2.816416e+09",
        planned_action="Convert to string and validate length"
    )

## 6. Check for missingness

In [ ]:
for name, df in simulated_providers.items():
    print(f"\n{name} Missingness")
    
    missing_summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })
    
    display(
        missing_summary[
            missing_summary["missing_count"] > 0
        ].sort_values("missing_pct", ascending=False)
    )

## 7. Check for exact row duplicates

In [ ]:
# No duplicated rows found
for name, df in simulated_providers.items():
    print(df[df.duplicated(keep=False)])

## 8. Duplicates and missingness check for each data source

In [179]:
npi_missingness = {
    "EHR": 7.93,
    "HR": 53.60,
    "Credentialing": 2.00
}

for dataset, pct in npi_missingness.items():
    add_finding(
        dataset=dataset,
        column="npi",
        issue="NPI contains missing values",
        example=f"{pct}% missing",
        planned_action="Retain missing values, do not impute NPI, use other attributes for matching",
    )

### EHR

In [ ]:
ehr_raw[ehr_raw["ehr_provider_id"].duplicated(keep=False)].sort_values("ehr_provider_id")
ehr_raw[ehr_raw["npi"].duplicated(keep=False)].sort_values("npi")

# Phone contains duplicates
ehr_raw[ehr_raw["phone"].duplicated(keep=False)].sort_values("phone")

In [ ]:
# inspecting duplciate phone records
phone_counts = ehr_raw["phone"].value_counts()
display(phone_counts[phone_counts > 1])

# Phone numbers are shared across multiple records
# Retain values and use as supporting match attribute only
add_finding(
    dataset="EHR",
    column="phone",
    issue="Phone numbers are shared across multiple provider records",
    example="(210) 617-5300 appears on 9 records",
    planned_action="Retain values and use as supporting match attribute only"
)

In [ ]:
ehr_raw[ehr_raw["zip"].duplicated(keep=False)].sort_values("zip")

zip_counts_ehr = ehr_raw["zip"].value_counts()
display(zip_counts_ehr[zip_counts_ehr > 1])

# Zip codes are shared across multiple records
# Retain values and use as supporting match attribute only
add_finding(
    dataset="EHR",
    column="zip",
    issue="Zip codes are shared across multiple provider records",
    example="782294402 appears on 15 records",
    planned_action="Retain values and use as supporting match attribute only"
)

In [ ]:
ehr_raw["zip"].dropna().astype(str).str.len().value_counts()

# Zip has mixed 5-digit and 9-digit formats
# Standardize ZIP format and derive 5-digit ZIP for matching
add_finding(
    dataset='EHR',
    column="zip",
    issue="ZIP has mixed 5-digit and 9-digit formats",
    example="77380, 774336767",
    planned_action="Standardize ZIP format and derive 5-digit ZIP for matching"
)

In [ ]:
# 19 records missing npi and phone
ehr_raw[
    ehr_raw["npi"].isna() &
    ehr_raw["phone"].isna()
]

# 0 records missing npi, phone and first_name
ehr_raw[
    ehr_raw["npi"].isna() &
    ehr_raw["phone"].isna() &
    ehr_raw["first_name"].isna() 
]

### HR

In [ ]:
# inspecting duplciate phone records
hr_raw.head()

In [ ]:
hr_raw[hr_raw["npi"].duplicated(keep=False)].sort_values("npi")
hr_raw[hr_raw["employee_id"].duplicated(keep=False)].sort_values("employee_id")

In [ ]:
phone_counts = hr_raw["work_phone"].value_counts()
display(phone_counts[phone_counts > 1])

# Phone numbers are shared across multiple records
# Retain values and use as supporting match attribute only
add_finding(
    dataset="HR",
    column="work_phone",
    issue="Phone numbers are shared across multiple provider records",
    example="713-792-6161 appears on 9 records",
    planned_action="Retain values and use as supporting match attribute only"
)

In [ ]:
# 151 records missing npi and work_phone
hr_raw[
    hr_raw["npi"].isna() &
    hr_raw["work_phone"].isna()
]

# 0 records missing npi, work_phone and first_name
hr_raw[
    hr_raw["npi"].isna() &
    hr_raw["work_phone"].isna() &
    hr_raw["first_name"].isna() 
]

### CREDENTIALING

In [ ]:
cred_raw.head()

In [ ]:
cred_raw[cred_raw["npi"].duplicated(keep=False)].sort_values("npi")
cred_raw[cred_raw["credentialing_id"].duplicated(keep=False)].sort_values("credentialing_id")
cred_raw[cred_raw["license_number"].duplicated(keep=False)].sort_values("license_number")
cred_raw[cred_raw["zip"].duplicated(keep=False)].sort_values("zip")

In [ ]:
zip_counts_cred = cred_raw["zip"].value_counts()
display(zip_counts_cred[zip_counts_cred > 1])

# Zip codes are shared across multiple records
# Retain values and use as supporting match attribute only
add_finding(
    dataset="Credentialing",
    column="zip",
    issue="Zip codes are shared across multiple provider records",
    example="782344504 appears on 13 records",
    planned_action="Retain values and use as supporting match attribute only"
)

In [ ]:
cred_raw.columns

In [ ]:
# 4 records missing npi and work_phone
cred_raw[
    cred_raw["npi"].isna() &
    cred_raw["phone"].isna()
]

# 0 records missing npi, work_phone and first_name
cred_raw[
    cred_raw["npi"].isna() &
    cred_raw["phone"].isna() &
    cred_raw["legal_first_name"].isna() 
]

## 9. Whitespace, hidden character and formatting check 

### EHR

In [ ]:
# Inspecting values for whitespaces
ehr_raw["first_name"].dropna().apply(repr).head(20)
ehr_raw["middle_name"].dropna().apply(repr).head(20)
ehr_raw["last_name"].dropna().apply(repr).head(20)
ehr_raw["credential"].dropna().apply(repr).head(20)
ehr_raw["city"].dropna().apply(repr).head(20)
ehr_raw["state"].dropna().apply(repr).head(20)
ehr_raw["address_line_1"].dropna().apply(repr).head(20)
ehr_raw["address_line_2"].dropna().apply(repr).head(20)
ehr_raw["specialty_code"].dropna().apply(repr).head(20)

In [ ]:
ehr_raw.columns

In [ ]:
# Checking for all possible lengths
ehr_raw["middle_name"].str.len().value_counts().sort_index()

In [ ]:
# Checking for capitalisation
ehr_raw["city"].value_counts().tail(30)
ehr_raw["credential"].value_counts().tail(30)
ehr_raw["specialty_code"].value_counts().tail(30)
ehr_raw["zip"].value_counts().head(30)
ehr_raw["phone"].value_counts().head(30)
ehr_raw["state"].value_counts().head(30)

### HR

In [ ]:
# Inspecting values for whitespaces
hr_raw["first_name"].dropna().apply(repr).head(20)
hr_raw["middle_initial"].dropna().apply(repr).tail(20)
hr_raw["last_name"].dropna().apply(repr).head(20)
hr_raw["job_credential"].dropna().apply(repr).head(20)
hr_raw["work_city"].dropna().apply(repr).head(20)
hr_raw["work_state"].dropna().apply(repr).head(20)
hr_raw["job_specialty_code"].dropna().apply(repr).head(20)
hr_raw["work_phone"].dropna().apply(repr).head(20)

In [ ]:
# Checking for all possible lengths
hr_raw["job_specialty_code"].str.len().value_counts().sort_index()
hr_raw["middle_initial"].str.len().value_counts().sort_index()

In [ ]:
# Checking for inconsistent abbreviations
sorted(hr_raw["work_city"].dropna().unique())
sorted(hr_raw["work_state"].dropna().unique())

### CREDENTIALING

In [ ]:
cred_raw.columns

In [ ]:
# Inspecting values for whitespaces
cred_raw["legal_first_name"].dropna().apply(repr).tail(20)
cred_raw["legal_middle_name"].dropna().apply(repr).head(20)
cred_raw["legal_last_name"].dropna().apply(repr).tail(20)
cred_raw["license_number"].dropna().apply(repr).head(20)
cred_raw["license_state"].dropna().apply(repr).head(20)
cred_raw["city"].dropna().apply(repr).head(20)
cred_raw["state"].dropna().apply(repr).head(20)
cred_raw["address_line_1"].dropna().apply(repr).head(20)
cred_raw["address_line_2"].dropna().apply(repr).tail(20)

In [ ]:
# Checking for all possible lengths
cred_raw["legal_middle_name"].str.len().value_counts().sort_index()
cred_raw["taxonomy_code"].str.len().value_counts().sort_index()
cred_raw["license_number"].str.len().value_counts().sort_index()
cred_raw["license_state"].str.len().value_counts().sort_index()


In [ ]:
# Checking for inconsistent abbreviations
sorted(cred_raw["city"].dropna().unique())
sorted(cred_raw["address_line_2"].dropna().unique())

In [ ]:
# EHR Findings

# first_name, middle_name, last_name
# Inconsistent capitalization
# Bray, TAYLOR

# legal_middle_name
# A lot of records have name length 1 character, while other records have more than 1 character

# credential
# Some records contain multiple credentials seperated by commas
# APRN, AGACNP-BC

# credential
# Inconsistent abbreviations
# MD, M.D.

# address_line_1
# Inconsistent abbreviations exist 
# 26006 OAKRIDGE DR., 4502 MEDICAL DR

# address_line_2
# Inconsistent suite representations and abbreviations
# SUITE 100, STE # 208, SUITE # 9, SUITE C-106, STE. 300

In [ ]:
# HR Findings

# middle_initial
# Mose records have intial of 1 character, while some records have more than 1 character

# credential
# Inconsistent abbreviations
# MD, M.D.

# work_city
# Commas in city field
# 'AUSTIN', 'AUSTIN,'

# work_city
# Inconsistent abbreviations for same city
# FT WORTH, FT. WORTH and LACKLAND A F B, LACKLAND AFB and MOUNT PLEASANT, MT PLEASANT

In [ ]:
# CRED Findings

# legal_first_name
# Found weird formatting
# CHIH- HAO

# legal_middle_name
# A lot of records have name length 1 character, while other records have more than 1 character

# credential
# Inconsistent abbreviations
# MD, M.D.

# license_number
# Inconsistent formatting and lengths, most have 5 characters
# 1-00-0350, 2000027672, 5912

# city
# Commas in city field
# 'AUSTIN', 'AUSTIN,'

# work_city
# Inconsistent abbreviations for same city
# FT WORTH, FT. WORTH and LACKLAND A F B, LACKLAND AFB and MOUNT PLEASANT, MT PLEASANT

# address_line_1
# Inconsistent abbreviations exist 
# 26006 OAKRIDGE DR., 4502 MEDICAL DR

# address_line_2
# Inconsistent suite representations and abbreviations
# SUITE 100, STE # 208, SUITE # 9, SUITE C-106, STE. 300

In [ ]:
# NAME FINDINGS

# Inconsistent capitalization in EHR name fields
for column in ["first_name", "middle_name", "last_name"]:
    add_finding(
        dataset="EHR",
        column=column,
        issue="Inconsistent capitalization",
        example="Bray, TAYLOR",
        planned_action="Trim whitespace and standardize capitalization"
    )


# Middle name fields contain both initials and full middle names
for dataset, column in [
    ("EHR", "legal_middle_name"),
    ("Credentialing", "legal_middle_name")
]:
    add_finding(
        dataset=dataset,
        column=column,
        issue="Mixed middle initials and full middle names",
        example="J, MARIE",
        planned_action="Preserve original value and standardize middle name representation for matching"
    )


# HR middle_initial contains values longer than one character
add_finding(
    dataset="HR",
    column="middle_initial",
    issue="Middle initial contains values longer than one character",
    example="J, MARIE",
    planned_action="Inspect multi-character values and derive standardized middle initial where appropriate"
)


# Unusual spacing around hyphenated first names
add_finding(
    dataset="Credentialing",
    column="legal_first_name",
    issue="Inconsistent spacing in hyphenated names",
    example="CHIH- HAO",
    planned_action="Normalize whitespace around hyphens while preserving hyphenated name"
)

# CREDENTIAL FINDINGS

# Credential abbreviations represented inconsistently
for dataset in ["EHR", "HR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="credential",
        issue="Inconsistent credential abbreviations",
        example="MD, M.D.",
        planned_action="Standardize equivalent credential abbreviations to a canonical format"
    )

# Multiple credentials stored in one field
add_finding(
    dataset="EHR",
    column="credential",
    issue="Multiple credentials stored in a single field",
    example="APRN, AGACNP-BC",
    planned_action="Parse multiple credentials and standardize each credential separately"
)

# CITY FINDINGS

# Trailing punctuation in city values
for dataset, column in [
    ("HR", "work_city"),
    ("Credentialing", "city")
]:
    add_finding(
        dataset=dataset,
        column=column,
        issue="Trailing punctuation in city values",
        example="AUSTIN,",
        planned_action="Remove trailing punctuation and trim whitespace"
    )


# Multiple representations of the same city
for dataset, column in [
    ("HR", "work_city"),
    ("Credentialing", "work_city")
]:
    add_finding(
        dataset=dataset,
        column=column,
        issue="Inconsistent city abbreviations",
        example="FT WORTH, FT. WORTH; LACKLAND A F B, LACKLAND AFB; MOUNT PLEASANT, MT PLEASANT",
        planned_action="Map known city variants to a consistent canonical city name"
    )

# ADDRESS FINDINGS

# Street suffix abbreviations vary
for dataset in ["EHR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="address_line_1",
        issue="Inconsistent street suffix abbreviations",
        example="26006 OAKRIDGE DR., 4502 MEDICAL DR",
        planned_action="Standardize street suffix abbreviations and remove unnecessary punctuation"
    )

# Suite/unit formatting varies
for dataset in ["EHR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="address_line_2",
        issue="Inconsistent suite formatting",
        example="SUITE 100, STE # 208, SUITE # 9, SUITE C-106, STE. 300",
        planned_action="Standardize suite/unit labels, punctuation, and spacing"
    )

# LICENSE FINDINGS

add_finding(
    dataset="Credentialing",
    column="license_number",
    issue="Inconsistent license number formatting and length",
    example="1-00-0350, 2000027672, 5912",
    planned_action="Convert to string, trim whitespace, and preserve valid formatting for state-specific validation"
)

# PHONE FINDINGS

# Phone numbers exist in varied formats
add_finding(
    dataset="EHR",
    column="phone",
    issue="Phone numbers exist in varied formats",
    example="(210) 617-5300, 8774182978",
    planned_action="Standardise phone formats",
)

# Phone numbers exist in varied formats
add_finding(
    dataset="HR",
    column="work_phone",
    issue="Phone numbers exist in varied formats",
    example="817-702-1244, 7137911414",
    planned_action="Inspect and standardise phone formats"
)

# ZIP CODE CHECK

# Zip has mixed 5-digit and 9-digit formats
add_finding(
    dataset='Credentialing',
    column="zip",
    issue="ZIP has mixed 5-digit and 9-digit formats",
    example="77380, 774336767",
    planned_action="Standardize ZIP format and derive 5-digit ZIP for matching"
)

In [ ]:
findings

## 10. Profiling across three source systems

In [ ]:
# Creating a cross system field map
equivalent_fields = {
    "first_name": {
        "EHR": "first_name",
        "HR": "first_name",
        "Credentialing": "legal_first_name"
    },
    "middle_name": {
        "EHR": "middle_name",
        "HR": "middle_initial",
        "Credentialing": "legal_middle_name"
    },
    "last_name": {
        "EHR": "last_name",
        "HR": "last_name",
        "Credentialing": "legal_last_name"
    },
    "credential": {
        "EHR": "credential",
        "HR": "job_credential",
        "Credentialing": "credential"
    },
    "specialty": {
        "EHR": "specialty_code",
        "HR": "job_specialty_code",
        "Credentialing": "taxonomy_code"
    },
    "city": {
        "EHR": "city",
        "HR": "work_city",
        "Credentialing": "city"
    },
    "state": {
        "EHR": "state",
        "HR": "work_state",
        "Credentialing": "state"
    },
    "phone": {
        "EHR": "phone",
        "HR": "work_phone",
        "Credentialing": "phone"
    }
}

datasets = {
    "EHR": ehr_raw,
    "HR": hr_raw,
    "Credentialing": cred_raw
}

In [ ]:
# Helper Functions

# Comparing equivalent columns to check for missingness, duplicates and uniqueness
comparison_rows = []

for concept, mappings in equivalent_fields.items():
    
    for dataset_name, column_name in mappings.items():
        df = datasets[dataset_name]

        comparison_rows.append({
            "concept": concept,
            "dataset": dataset_name,
            "column": column_name,
            "dtype": df[column_name].dtype,
            "missing_count": df[column_name].isna().sum(),
            "missing_pct": round(df[column_name].isna().mean() * 100, 2),
            "unique_count": df[column_name].nunique(dropna=True),
            "sample_values": df[column_name].dropna().astype(str).unique()[:5]
        })

# Comparing individual field values
def compare_field_values(concept, n=40):
    mappings = equivalent_fields[concept]

    for dataset_name, column_name in mappings.items():
        print(f"\n{dataset_name} - {column_name}")
        print("-" * 40)

        print(
            datasets[dataset_name][column_name]
            .dropna()
            .astype(str)
            .value_counts()
            .head(n)
        )

# Comparing string lengths
def compare_lengths(concept):
    mappings = equivalent_fields[concept]

    for dataset_name, column_name in mappings.items():

        lengths = (
            datasets[dataset_name][column_name]
            .dropna()
            .astype(str)
            .str.len()
        )

        print(f"\n{dataset_name} - {column_name}")
        print(lengths.value_counts().sort_index())

# Comparing across sources excluding missing values
def compare_across_sources(
    ehr_column,
    hr_column,
    cred_column,
    output_name
):
    
    ehr_compare = (
        ehr_raw.loc[ehr_raw["npi"].notna(), ["npi", ehr_column]]
        .rename(columns={ehr_column: f"ehr_{output_name}"})
    )
    
    hr_compare = (
        hr_raw.loc[hr_raw["npi"].notna(), ["npi", hr_column]]
        .rename(columns={hr_column: f"hr_{output_name}"})
    )
    
    cred_compare = (
        cred_raw.loc[cred_raw["npi"].notna(), ["npi", cred_column]]
        .rename(columns={cred_column: f"cred_{output_name}"})
    )
    
    comparison = (
        ehr_compare
        .merge(
            hr_compare,
            on="npi",
            how="inner",
            validate="one_to_one"
        )
        .merge(
            cred_compare,
            on="npi",
            how="inner",
            validate="one_to_one"
        )
    )
    
    return comparison

In [ ]:
# Comparing equivalent columns to check for missingness, duplicates and uniqueness
field_comparison = pd.DataFrame(comparison_rows)

field_comparison

In [ ]:
compare_field_values("credential")


In [ ]:
compare_lengths("middle_name")

In [ ]:
# Comparing first_name across sources
first_name_compare = compare_across_sources(
    "first_name",
    "first_name",
    "legal_first_name",
    "first_name"
)

first_name_compare.head()

In [ ]:
# Comparing credential across sources
credential_compare = compare_across_sources(
    "credential",
    "job_credential",
    "credential",
    "credential"
)

credential_compare.tail(40)

In [ ]:
# Comparing specialty across sources
specialty_compare = compare_across_sources(
    "specialty_code",
    "job_specialty_code",
    "taxonomy_code",
    "specialty"
)

specialty_compare.tail(20)

In [ ]:
# Comparing city across sources
specialty_compare = compare_across_sources(
    "city",
    "work_city",
    "city",
    "city"
)

specialty_compare.tail(40)

In [180]:
# credential in credentialing
# some multiple credentials are not seperated by commas 
add_finding(
    dataset="CREDENTIALING",
    column="credential",
    issue="Multiple credentials stored with inconsistent separators",
    example="M.S. CCC-SLP",
    planned_action="Preserve raw value and parse recognized credentials into a standardized representation"
)

# credential in hr, ehr and cred
# inconsistent abbreviations
add_finding(
    dataset="EHR / HR / CREDENTIALING",
    column="credential / job_credential",
    issue="Inconsistent credential abbreviations",
    example="O.D. | OD",
    planned_action="Map credential variants to canonical credential values"
)

# speciality missing for hr source, but available in other 2 spurces
add_finding(
    dataset="HR",
    column="job_specialty_code",
    issue="Specialty code contains missing values",
    example="21.38% missing",
    planned_action="Retain missing values; use available specialty data from other sources during survivorship",
)

Finding already exists.
Finding already exists.


## 11. Saving findings log to source_profiling_findings

In [ ]:
# Save profiling findings

PROFILE_DIR = Path("../data/processed/profiling")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

FINDINGS_PATH = PROFILE_DIR / "source_profiling_findings.csv"

findings_to_save = (
    findings
    .drop_duplicates(subset=["dataset", "column", "issue"])
    .sort_values(["dataset", "column", "issue"])
    .reset_index(drop=True)
)

findings_to_save.to_csv(
    FINDINGS_PATH,
    index=False
)

print(f"Saved {len(findings_to_save)} findings to:")
print(FINDINGS_PATH)